# Tetrahedral MH and MH²M

Native volume and facet assembly with physical Neumann data. This notebook verifies an exact affine problem and displays a separately acquired nonaffine five-level campaign. The experiment extends the MH Robin formulation to three dimensions and uses the three-dimensional MH²M local maps; it is not a numerical table from either paper.

In [ ]:
from pathlib import Path
import json
import numpy as np
from IPython.display import Image, display
from threadpoolctl import threadpool_limits
from pymhm.tetrahedral import TetraMesh
from pymhm.darcy3d import TriangularSkeleton
from pymhm.mh3d import solve_mh_3d
from pymhm.mh2m3d import solve_mh2m_3d

ROOT = Path.cwd()
if not (ROOT / "examples").exists():
    ROOT = ROOT.parent

For $K=I$, let $p=1+x+y+z$, $q=(-1,-1,-1)$ and $f=0$. Prescribe the outward physical flux on every boundary face and the exact volume mean $5/2$. The MH Robin multiplier differs from this flux by $p\sigma\cdot n$ and therefore needs degree one for this affine pressure, even though the physical normal flux is constant. The API applies that conversion through its boundary equations.

In [ ]:
mesh = TetraMesh.unit_cube()
neumann = {int(face): -float(mesh.normals[face].sum()) for face in mesh.boundary_faces}
with threadpool_limits(1):
    for solve, options in ((solve_mh_3d, {"skeleton": TriangularSkeleton(mesh, degree=1)}),
                           (solve_mh2m_3d, {})):
        result = solve(mesh, neumann=neumann, mean_pressure=2.5, **options)
        p_error = result.l2_error(lambda x: 1 + x.sum(axis=1), order=4)
        q_error = result.flux_l2_error([-1.0, -1.0, -1.0], order=4)
        balance = np.max(np.abs(result.conservation_residuals()))
        assert max(p_error, q_error, balance) < 2e-10
        print(solve.__name__, {"pressure_L2": p_error, "flux_L2": q_error, "macro_balance": balance})

The archived refinement uses $p=1+x+2y+3z+\sin(\pi x)\sin(\pi y)\sin(\pi z)$, $K=I$, and nonzero Dirichlet data. MH uses local $P_3$ and $P_1$ Robin faces; MH²M uses local $P_2$, continuous $\Gamma=P_2$ and subdivided broken $\Lambda=P_1$. Errors are genuine volume integrals with independent orders 6 and 8. The two configurations have different local and global dimensions.

In [ ]:
record = json.loads((ROOT / "examples/results/mh3d/comparison.json").read_text())
assert record["source_changed_during_run"] is False
assert len(record["rows"]) == 10
for row in record["rows"]:
    print(row["method"], row["resolution"], row["global_unknowns"], row["pressure_relative"], row["flux_relative"])
display(Image(filename=str(ROOT / "docs/figures/mh3d/convergence.png")))

The section plots use one-sided fine-cell samples, with the actual macroface intersections in every panel. The signed $q_x$ component preserves direction; no averaging is applied across macrointerfaces.

In [ ]:
for name in ("mh-fields.png", "mh2m-fields.png"):
    display(Image(filename=str(ROOT / "docs/figures/mh3d" / name)))